# Aula 11 — Classificação, Clusterização e Introdução a Machine Learning
**Ciência de Dados · Univassouras · Prof. Mesc. Diego Ramos Inácio**

---

> **Conexão com a Aula 10:** aprendemos a diagnosticar e avaliar modelos com métricas avançadas (ROC, AUC, validação cruzada).
> Agora expandimos o repertório: conhecemos quatro algoritmos de classificação, dois de clusterização,
> e montamos um pipeline de ML completo com busca de hiperparâmetros.

| # | Conteúdo |
|---|----------|
| 0 | Configuração do ambiente |
| 1 | **Exemplo 1** — KNN, Árvore de Decisão, Random Forest e SVM: comparativo |
| 2 | **Exemplo 2** — K-Means: segmentação de clientes |
| 3 | **Exemplo 3** — DBSCAN: clusters de formato livre e detecção de outliers |
| 4 | **Exemplo 4** — Pipeline ML completo com GridSearchCV |
| 5 | **Exercício** — sua vez! |

> **Dica:** execute cada célula com `Shift + Enter`.

## 0 · Configuração do Ambiente

Instale as dependências caso ainda não tenha (remova o `#` e execute).

In [ ]:
# !pip install numpy pandas matplotlib scikit-learn scipy --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Classificação
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Clusterização
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors

# Utilitários
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    GridSearchCV, StratifiedKFold
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    roc_auc_score, ConfusionMatrixDisplay,
    silhouette_score, silhouette_samples
)
from sklearn.decomposition import PCA

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('Ambiente pronto!')

---
## 1 · Exemplo 1 — Comparativo de Algoritmos de Classificação
### Problema: prever fraude em transações bancárias

**Variável alvo (Y):** Transação fraudulenta (1) ou legítima (0)  
**Preditores (X):** valor da transação, hora do dia, distância do comerciante (km), número de tentativas no dia, saldo disponível

---

### Os quatro algoritmos desta aula — intuição antes do código

#### KNN — K-Nearest Neighbors (K Vizinhos Mais Próximos)

**Analogia:** imagine que você está numa cidade nova e quer saber se um bairro é seguro. Você pergunta às **K pessoas mais próximas** de você e vota pela opinião da maioria.

O KNN funciona exatamente assim:
1. Para cada transação nova, calcula a **distância** dela para todas as transações já conhecidas
2. Seleciona as **K mais próximas** (os "vizinhos")
3. Verifica qual classe é mais comum entre essas K transações — essa é a previsão

> **Por que precisa de padronização?**  
> Se uma variável está em R$ (0 a 10.000) e outra em km (0 a 100), a distância fica dominada pelo R$ — é como comparar metros com quilômetros sem converter. O `StandardScaler` coloca tudo na mesma escala.

> **Como escolher K?**  
> - K muito pequeno (ex: K=1): o modelo "decora" os dados de treino — overfitting  
> - K muito grande (ex: K=500): a fronteira fica tão suave que ignora padrões reais — underfitting  
> - Prática: testar K = 3, 5, 7, 9... via validação cruzada

---

#### Arvore de Decisão — Decision Tree

**Analogia:** é como um fluxograma de perguntas de "sim ou não". Você vai respondendo e chega a uma conclusão.

```
"A transação foi feita de madrugada (hora < 6)?"
       +-- NAO --> "O valor é maior que R$1.500?"
       |              +-- NAO --> Provavelmente legítima
       |              +-- SIM --> Suspeito
       +-- SIM --> "Tentativas > 3 no dia?"
                      +-- NAO --> Suspeito
                      +-- SIM --> Alta chance de fraude
```

O algoritmo aprende **automaticamente** quais perguntas fazer e em que ordem, escolhendo as divisões que mais separam as classes. A medida de "pureza" usada é o **Gini**:
- Gini = 0 — grupo puro (todos da mesma classe)
- Gini = 0,5 — grupo misturado pela metade

> **Risco de overfitting:** sem limitar a profundidade, a árvore pode criar uma regra para cada ponto do treino. Use `max_depth` para controlar.

---

#### Random Forest — Floresta Aleatória

**Analogia:** em vez de consultar um único especialista, você consulta **200 especialistas diferentes** e adota a opinião da maioria.

Como cada árvore é diferente:
- Cada uma treina em uma **amostra aleatória** dos dados (bootstrap: sorteio com reposição)
- Cada divisão considera apenas um **subconjunto aleatório** das variáveis

Isso garante **diversidade**: as árvores erram em momentos diferentes, e quando somadas, os erros se cancelam.

> **Feature Importance:** o Random Forest mede naturalmente quais variáveis foram mais usadas nas divisões — uma ferramenta poderosa para entender os dados.

---

#### SVM — Support Vector Machine (Máquina de Vetor de Suporte)

**Analogia:** imagine duas nuvens de pontos (fraude e legítima) numa folha de papel. O SVM encontra a **linha que as separa com a maior distância possível** de ambos os lados — como colocar uma régua entre elas com a maior folga.

Os pontos mais próximos dessa linha são os **vetores de suporte** — são eles que definem e sustentam a fronteira.

**Kernel RBF:** quando as classes não são separáveis por uma linha reta, o SVM usa o **kernel RBF** para "dobrar" o espaço e tornar a separação possível — como amassar uma folha de papel para que dois grupos que estavam embaralhados agora fiquem em lados opostos.

---

| Algoritmo | Analogia simples | Precisa de padronização? |
|-----------|-----------------|--------------------------|
| **KNN** | Votação por vizinhança | Obrigatória (usa distância) |
| **Arvore** | Fluxograma de perguntas | Nao necessária |
| **Random Forest** | Votação de 200 árvores | Nao necessária |
| **SVM** | Linha de máxima margem | Obrigatória (usa geometria) |

In [ ]:
# ── Dados de fraude bancária ───────────────────────────────────
n1 = 800
valor_tx       = np.random.exponential(500, n1)          # valor da transação (R$)
hora_dia       = np.random.randint(0, 24, n1)            # hora (0–23)
dist_km        = np.random.exponential(30, n1)            # distância do comerciante
tentativas     = np.random.randint(1, 8, n1)              # tentativas no dia
saldo_disp     = np.random.uniform(0, 10000, n1)          # saldo disponível

# Lógica real de fraude
logit1 = (-4
           + 0.003 * valor_tx
           + 0.08  * tentativas
           + 0.04  * dist_km
           - 0.0003 * saldo_disp
           + 0.04  * (hora_dia < 6).astype(int))   # madrugada aumenta risco
prob1  = 1 / (1 + np.exp(-logit1))
fraude = (np.random.uniform(0, 1, n1) < prob1).astype(int)

df1 = pd.DataFrame({
    'valor_tx':   valor_tx.round(2),
    'hora_dia':   hora_dia,
    'dist_km':    dist_km.round(1),
    'tentativas': tentativas,
    'saldo_disp': saldo_disp.round(2),
    'fraude':     fraude
})

print('Distribuição das classes:')
print(df1['fraude'].value_counts().rename({0: 'Legítima (0)', 1: 'Fraude (1)'}))
print(f'\nTaxa de fraude: {df1["fraude"].mean():.1%}')
print('\nEstatísticas descritivas:')
df1.describe().round(2)

In [ ]:
# ── Exploração visual ──────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
variaveis1 = ['valor_tx', 'hora_dia', 'dist_km', 'tentativas', 'saldo_disp']
cores_cls  = {0: '#2563eb', 1: '#e11d48'}

for idx, var in enumerate(variaveis1):
    ax = axes.flatten()[idx]
    for cls in [0, 1]:
        label = 'Legítima' if cls == 0 else 'Fraude'
        ax.hist(df1[df1['fraude'] == cls][var], bins=25,
                alpha=0.65, color=cores_cls[cls], label=label, edgecolor='white')
    ax.set_xlabel(var.replace('_', ' ').title())
    ax.set_ylabel('Frequência')
    ax.set_title(f'Distribuição: {var}')
    ax.legend(fontsize=9)

# Boxplot de tentativas por classe
ax_box = axes.flatten()[5]
legit  = df1[df1['fraude'] == 0]['tentativas']
fraud  = df1[df1['fraude'] == 1]['tentativas']
ax_box.boxplot([legit, fraud], labels=['Legítima', 'Fraude'],
               patch_artist=True,
               boxprops=dict(facecolor='#dbeafe'),
               medianprops=dict(color='#1d4ed8', lw=2))
ax_box.set_title('Tentativas por classe')
ax_box.set_ylabel('Tentativas no dia')

plt.suptitle('Exemplo 1 — Exploração dos dados de fraude bancária', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Treinar e comparar os 4 algoritmos via CV ──────────────────
X1 = df1.drop(columns='fraude')
y1 = df1['fraude']

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42, stratify=y1
)

modelos1 = {
    'KNN (k=7)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=7))
    ]),
    'Árvore (max_depth=5)': Pipeline([
        ('clf', DecisionTreeClassifier(max_depth=5, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('clf', RandomForestClassifier(n_estimators=200, random_state=42))
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', probability=True, random_state=42))
    ]),
}

cv1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados1 = {}

print(f'{'Algoritmo':30s} {'F1 médio':>10} {'± DP':>8} {'AUC médio':>11}')
print('-' * 65)

for nome, pipe in modelos1.items():
    f1s  = cross_val_score(pipe, X1_train, y1_train, cv=cv1, scoring='f1')
    aucs = cross_val_score(pipe, X1_train, y1_train, cv=cv1, scoring='roc_auc')
    resultados1[nome] = {'f1': f1s, 'auc': aucs}
    print(f'{nome:30s} {f1s.mean():>10.4f} {f1s.std():>8.4f} {aucs.mean():>11.4f}')

In [ ]:
# ── Visualização comparativa ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
nomes = list(resultados1.keys())
cores_box = ['#0d9488', '#2563eb', '#15803d', '#7c3aed']

for ax_idx, metrica in enumerate(['f1', 'auc']):
    valores = [resultados1[n][metrica] for n in nomes]
    bp = axes[ax_idx].boxplot(
        valores, labels=[n.split('(')[0].strip() for n in nomes],
        patch_artist=True, widths=0.5
    )
    for patch, cor in zip(bp['boxes'], cores_box):
        patch.set_facecolor(cor)
        patch.set_alpha(0.7)
    for median in bp['medians']:
        median.set_color('white')
        median.set_linewidth(2)
    label_m = 'F1-Score' if metrica == 'f1' else 'AUC'
    axes[ax_idx].set_ylabel(label_m)
    axes[ax_idx].set_title(f'{label_m} — 5-fold CV\n(caixa menor = estimativa mais estável)')
    axes[ax_idx].grid(axis='y', alpha=.3)
    axes[ax_idx].set_ylim(0.4, 1.05)
    axes[ax_idx].tick_params(axis='x', rotation=15)

plt.suptitle('Exemplo 1 — Comparativo de Algoritmos de Classificação', fontweight='bold')
plt.tight_layout()
plt.show()

# Avaliação do melhor modelo no conjunto de teste
melhor_nome = max(resultados1, key=lambda n: resultados1[n]['f1'].mean())
melhor_pipe = modelos1[melhor_nome]
melhor_pipe.fit(X1_train, y1_train)
y1_pred = melhor_pipe.predict(X1_test)

print(f'\nMelhor modelo (F1 CV): {melhor_nome}')
print('Relatório no conjunto de teste:')
print(classification_report(y1_test, y1_pred, target_names=['Legítima', 'Fraude']))

In [ ]:
# ── Visualizando a Árvore de Decisão e importância de variáveis ─
pipe_tree = modelos1['Árvore (max_depth=5)']
pipe_tree.fit(X1_train, y1_train)
tree_model = pipe_tree.named_steps['clf']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Árvore (3 primeiros níveis)
plot_tree(tree_model, feature_names=X1.columns.tolist(),
          class_names=['Legítima', 'Fraude'],
          max_depth=3, filled=True, ax=axes[0],
          fontsize=7, proportion=True,
          impurity=False, rounded=True)
axes[0].set_title('Árvore de Decisão (primeiros 3 níveis)', fontweight='bold')

# Importância de variáveis — Random Forest
pipe_rf = modelos1['Random Forest']
pipe_rf.fit(X1_train, y1_train)
rf_model = pipe_rf.named_steps['clf']
importancias = pd.Series(rf_model.feature_importances_, index=X1.columns)
importancias.sort_values().plot(kind='barh', ax=axes[1], color='#15803d', alpha=.8)
axes[1].set_title('Importância de Variáveis — Random Forest', fontweight='bold')
axes[1].set_xlabel('Feature Importance (Gini)')
axes[1].grid(axis='x', alpha=.3)

plt.tight_layout()
plt.show()

print('Importância de variáveis (Random Forest):')
print(importancias.sort_values(ascending=False).round(4).to_string())

### Lendo a Arvore de Decisão e a Importância de Variáveis

**Como ler o gráfico da árvore:**  
Cada nó (caixa) mostra:
- A **pergunta de divisão** (ex: `tentativas <= 3.5`)
- O **valor de Gini** — quão misturado está o grupo (0 = puro, 0.5 = metade de cada classe)
- A **proporção** de cada classe no nó

As folhas (nós finais, sem filhos) mostram a classe prevista. Quanto mais escura a cor, mais puro o nó.

**Como ler a Importância de Variáveis (Random Forest):**  
O Random Forest calcula, para cada variável, o quanto ela **reduziu a impureza (Gini)** ao longo de todas as 200 árvores. Uma variável com importância alta foi muito usada nas divisões — é relevante para separar as classes.

> Se uma variável tem importância próxima de zero, ela provavelmente não agrega informação útil ao modelo e poderia ser removida sem perda de desempenho (isso é chamado de **seleção de features**).

---
## 2 · Exemplo 2 — K-Means: Segmentação de Clientes
### Problema: agrupar clientes por comportamento de compra

**Dados (sem rótulo):** gasto mensal médio (R$), frequência de compras/mês, ticket médio (R$)

> **Objetivo:** descobrir grupos naturais de clientes para direcionar campanhas de marketing personalizadas.

---

### O que é clusterização e por que é diferente de classificação?

Na **classificação** (Exemplo 1), o modelo aprende com dados **já rotulados**: "esta transação É fraude, aquela NAO É". Existe uma resposta correta.

Na **clusterização**, os dados **nao têm rótulos**. Ninguém disse de antemão quais clientes são "premium" ou "econômicos". O algoritmo descobre por conta própria quais clientes são similares entre si e os agrupa.

É como separar uma gaveta de botões misturados por cor, tamanho e material — sem que ninguém te diga quantas categorias existem ou quais são.

---

### Como o K-Means funciona — passo a passo

O K-Means é um algoritmo **iterativo** (repete até convergir). Funciona assim:

```
Passo 0: Você decide K = 4 (quer encontrar 4 grupos)

Passo 1 — Inicialização: posiciona 4 centróides aleatoriamente no espaço
          (centróide = ponto central, a "média" do grupo)

Passo 2 — Atribuição: cada cliente vai para o centróide mais próximo
          cliente A --> Grupo 2 (centróide 2 é o mais perto)
          cliente B --> Grupo 1 (centróide 1 é o mais perto)
          ...

Passo 3 — Atualização: recalcula cada centróide como a média dos seus membros
          Centróide 1 = média de todos os clientes no Grupo 1
          Centróide 2 = média de todos os clientes no Grupo 2
          ...

Passo 4 — Repete os Passos 2 e 3 até os centróides pararem de se mover
          (convergência = os grupos estabilizaram)
```

> **Por que precisa de padronização?**  
> O K-Means usa **distância Euclidiana**. Se `gasto_mensal` vai até R$5.000 e `freq_compras` vai até 12, a variável de gasto domina completamente o cálculo. Um cliente que gasta pouco mas compra muito poderia ser jogado no cluster errado. O `StandardScaler` deixa todas as variáveis na mesma escala (média 0, desvio 1).

---

### Inércia (WCSS) — o que o algoritmo minimiza

A **inércia** (ou WCSS — *Within-Cluster Sum of Squares*) é a soma das distâncias quadráticas de cada ponto ao centróide do seu cluster:

```
WCSS = Soma de (distância de cada ponto ao seu centróide)²
```

- Inércia baixa = clusters compactos (pontos próximos ao centróide)
- Inércia alta = clusters dispersos
- Com K = número de pontos, inércia = 0 (cada ponto é seu próprio cluster — inútil)

---

### Silhouette Score — validando a qualidade dos clusters

O Silhouette mede, para cada ponto, **se ele está no cluster certo**:

```
silhouette(ponto) = (b - a) / max(a, b)

onde:
  a = distância média aos outros pontos do SEU cluster (coesão)
  b = distância média ao cluster MAIS PRÓXIMO (separação)
```

- **+1**: ponto muito bem alocado (longe dos outros clusters, perto do seu)
- **0**: ponto na fronteira entre dois clusters
- **-1**: ponto provavelmente no cluster errado

> O **Silhouette Score médio** é a média de todos os pontos — use em conjunto com o cotovelo para escolher K.

---

### O que é PCA? (Visualização 2D)

Nossos dados têm 3 dimensões (gasto, frequência, ticket). Para visualizar em 2D, usamos **PCA (Análise de Componentes Principais)**:

- PCA encontra as **direções de maior variação** nos dados
- Comprime as 3 dimensões originais em 2 novas dimensões (PC1 e PC2)
- PC1 captura a maior parte da variação, PC2 a segunda maior
- É uma simplificação visual — os clusters reais foram criados no espaço 3D original

> PC1 e PC2 nao têm interpretação direta ("nao é o gasto" nem "a frequência") — são combinações das variáveis originais.

---

| Etapa | O que faremos |
|-------|---------------|
| 1 | Padronização com StandardScaler |
| 2 | Método do cotovelo (WCSS) para escolher K |
| 3 | Silhouette Score para confirmar K |
| 4 | Treinar K-Means e interpretar perfil dos clusters |
| 5 | Visualização 2D via PCA |

In [ ]:
# ── Dados de clientes (sem rótulos) ───────────────────────────
np.random.seed(7)
n2 = 400

# 4 grupos latentes (simulamos como se fossem grupos reais)
grupos_lat = np.random.choice([0, 1, 2, 3], n2, p=[0.3, 0.25, 0.25, 0.2])

params_grupos = {
    0: dict(gasto_m=(800, 200),  freq=(2, 1),   ticket=(350, 60)),   # baixo valor, baixa freq
    1: dict(gasto_m=(2500, 400), freq=(8, 2),   ticket=(320, 50)),   # alto freq, ticket médio
    2: dict(gasto_m=(5000, 800), freq=(4, 1),   ticket=(1200, 150)), # alto valor, baixa freq
    3: dict(gasto_m=(1500, 300), freq=(12, 3),  ticket=(130, 30)),   # freq muito alta, ticket baixo
}

gasto_mensal = np.array([np.random.normal(params_grupos[g]['gasto_m'][0],
                                           params_grupos[g]['gasto_m'][1]) for g in grupos_lat])
freq_compras = np.array([max(1, np.random.normal(params_grupos[g]['freq'][0],
                                                   params_grupos[g]['freq'][1])) for g in grupos_lat])
ticket_medio = np.array([np.random.normal(params_grupos[g]['ticket'][0],
                                           params_grupos[g]['ticket'][1]) for g in grupos_lat])

df2 = pd.DataFrame({
    'gasto_mensal_R$': gasto_mensal.clip(100).round(2),
    'freq_compras_mes': freq_compras.round(1),
    'ticket_medio_R$': ticket_medio.clip(20).round(2)
})

print('Dados de clientes — sem rótulo de grupo:')
print(df2.describe().round(2))

In [ ]:
# ── Padronização + Método do Cotovelo e Silhouette ─────────────
scaler2 = StandardScaler()
X2_s = scaler2.fit_transform(df2)

ks         = range(2, 11)
inercias   = []
silhouettes = []

for k in ks:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels_k = km.fit_predict(X2_s)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X2_s, labels_k))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Cotovelo
axes[0].plot(ks, inercias, 'o-', color='#2563eb', lw=2)
axes[0].axvline(4, color='red', ls='--', lw=1.5, label='K = 4 (cotovelo)')
axes[0].set_xlabel('Número de clusters (K)')
axes[0].set_ylabel('Inércia (WCSS)')
axes[0].set_title('Método do Cotovelo\n(ponto de inflexão = K ideal)')
axes[0].legend()
axes[0].grid(alpha=.3)

# Silhouette
axes[1].bar(ks, silhouettes, color='#0d9488', alpha=.8, edgecolor='white')
best_k = list(ks)[np.argmax(silhouettes)]
axes[1].axvline(best_k, color='red', ls='--', lw=1.5,
                label=f'K = {best_k} (maior silhouette)')
axes[1].set_xlabel('Número de clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score por K\n(maior = clusters mais compactos e separados)')
axes[1].legend()
axes[1].grid(axis='y', alpha=.3)

plt.suptitle('Exemplo 2 — Escolha do K ótimo', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Silhouette por K: { {k: round(s, 4) for k, s in zip(ks, silhouettes)} }')
print(f'K com maior silhouette: {best_k}')

In [ ]:
# ── K-Means com K = 4 e interpretação ─────────────────────────
km_final = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42)
df2['cluster'] = km_final.fit_predict(X2_s)

print('Perfil médio de cada cluster:')
print(df2.groupby('cluster')[['gasto_mensal_R$', 'freq_compras_mes', 'ticket_medio_R$']]
        .mean().round(2))

print('\nTamanho de cada cluster:')
print(df2['cluster'].value_counts().sort_index())

# Visualização via PCA 2D
pca = PCA(n_components=2, random_state=42)
X2_pca = pca.fit_transform(X2_s)

cores2   = ['#0d9488', '#2563eb', '#e11d48', '#d97706']
nomes_cl = ['Cluster 0', 'Cluster 1', 'Cluster 2', 'Cluster 3']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# PCA 2D
for cl in range(4):
    mask = df2['cluster'] == cl
    axes[0].scatter(X2_pca[mask, 0], X2_pca[mask, 1],
                    color=cores2[cl], alpha=.6, s=35, label=nomes_cl[cl])
centroids_pca = pca.transform(km_final.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                color='black', s=200, marker='X', zorder=5, label='Centróides')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variância)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variância)')
axes[0].set_title('Clusters no espaço PCA 2D')
axes[0].legend(fontsize=9)

# Gasto vs Ticket colorido por cluster
for cl in range(4):
    mask = df2['cluster'] == cl
    axes[1].scatter(df2[mask]['gasto_mensal_R$'], df2[mask]['ticket_medio_R$'],
                    color=cores2[cl], alpha=.5, s=35, label=nomes_cl[cl])
axes[1].set_xlabel('Gasto Mensal (R$)')
axes[1].set_ylabel('Ticket Médio (R$)')
axes[1].set_title('Gasto Mensal vs Ticket Médio')
axes[1].legend(fontsize=9)

plt.suptitle('Exemplo 2 — K-Means: Segmentação de Clientes (K=4)', fontweight='bold')
plt.tight_layout()
plt.show()

sil_final = silhouette_score(X2_s, df2['cluster'])
print(f'\nSilhouette Score final (K=4): {sil_final:.4f}')

---
## 3 · Exemplo 3 — DBSCAN: Clusters de Formato Livre
### Por que o K-Means falha em alguns casos?

---

### O que é o formato de "meia lua"?

O dataset `make_moons` gera dois grupos de pontos em formato de **meias-luas entrelaçadas** — pense em duas fatias de laranja encaixadas uma na outra, ou dois crescentes em espelho:

```
        oooooo                    (grupo 0 — meia-lua superior)
      oo      oo
    oo          oo
  oo              oo
    oo          oo
      oo      oo
  xxxxxxxxxxxxxxxx                (grupo 1 — meia-lua inferior)
```

Esse formato é um **teste clássico** para avaliar algoritmos de clusterização, porque:
- Os dois grupos **se curvam ao redor um do outro**
- Nenhuma linha reta consegue separá-los completamente
- Mas qualquer humano consegue identificar os dois grupos visualmente sem esforço

---

### Por que o K-Means falha nesse formato?

O K-Means funciona assim: calcula um **centróide** (ponto médio) para cada grupo, e atribui cada ponto ao centróide mais próximo. O pressuposto implícito é que clusters são **esféricos** — bolas ao redor de um centro.

Nas meias-luas, o centróide de cada grupo fica **dentro da curvatura** — ou seja, num lugar onde quase nenhum ponto do grupo realmente está. O resultado é que a linha divisória corta **no meio das meias-luas**, misturando os dois grupos.

```
K-Means ve:            O que queremos:

  oo| xxxxxxx          oooooo | xxxxxxx
 oo |   xxxx           oo    |   xxxx
oo  |     xxx         oo     |     xxx
 oo |   xxxx           oo    |   xxxx
  oo| xxxxxxx          oooooo | xxxxxxx
    |
(linha errada)      (o humano separa corretamente)
```

---

### Como o DBSCAN resolve?

DBSCAN nao usa centróides. Em vez disso, usa o conceito de **densidade**: dois pontos fazem parte do mesmo cluster se há uma cadeia de pontos densamente conectados entre eles.

**Analogia:** imagine uma multidao num parque. Você começa com uma pessoa qualquer e pergunta: "tem alguém a menos de 2 metros?" Se sim, essas pessoas também fazem parte do mesmo grupo. Você repete isso para os vizinhos, e assim vai "caminhando" pelo grupo sem precisar saber sua forma.

Dessa forma, o DBSCAN consegue **seguir a curvatura** da meia-lua — nao importa a forma, ele vai conectando pontos densos até percorrer o grupo inteiro.

---

### Os três tipos de ponto no DBSCAN

Para cada ponto, o DBSCAN avalia sua vizinhança dentro de um raio `eps`:

```
Exemplo com eps = raio de busca e min_samples = 4:

  [A] tem 5 vizinhos dentro do raio  --> CORE POINT  (forma o nucleo do cluster)
  [B] tem 2 vizinhos, mas é vizinho de A --> BORDER POINT (pertence ao cluster de A)
  [C] tem 0 vizinhos dentro do raio  --> NOISE POINT (label = -1, outlier)
```

| Tipo | Definição | Label atribuído |
|------|-----------|-----------------|
| **Core point** | Tem >= `min_samples` vizinhos dentro do raio `eps` | Número do cluster (0, 1, 2...) |
| **Border point** | É vizinho de um core point, mas nao tem vizinhos suficientes para ser core | Número do cluster do core mais próximo |
| **Noise point** | Nao é core nem border — está completamente isolado | **-1** (outlier) |

---

### Os dois parâmetros do DBSCAN

**`eps` (epsilon):** o raio de vizinhança. Define o que é "próximo".
- `eps` pequeno — muitos pontos isolados são ruído, clusters muito rígidos
- `eps` grande — tudo vira um cluster só

**`min_samples`:** número mínimo de vizinhos para ser um core point.
- `min_samples` pequeno — mais pontos viram core, menos ruído
- `min_samples` grande — exige densidade maior, mais ruído

> **Regra prática:** `min_samples` >= dimensões do dataset + 1. Para dados 2D, comece com `min_samples = 5`.  
> Para escolher `eps`, use o gráfico de k-vizinhos (veja a próxima célula).

---

### O label -1 é especial — e muito útil!

Pontos com label = -1 são **outliers detectados automaticamente** — dados que nao pertencem a nenhum grupo denso. Em aplicações reais, esses pontos merecem atenção especial:
- **Detecção de fraude:** transações isoladas que nao se encaixam em nenhum padrão
- **Saúde:** pacientes com perfil clínico muito atípico
- **Qualidade industrial:** peças com medidas fora do padrão de qualquer lote

In [ ]:
# ── Dados com clusters em formato de lua ───────────────────────
from sklearn.datasets import make_moons, make_blobs

np.random.seed(42)
X_moons, _ = make_moons(n_samples=300, noise=0.08, random_state=42)

# Adicionar outliers
outliers_xy = np.random.uniform(-1.5, 2.5, (20, 2))
X_with_out  = np.vstack([X_moons, outliers_xy])

scaler3 = StandardScaler()
X3_s    = scaler3.fit_transform(X_with_out)

# K-Means (vai falhar)
km3 = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_km3 = km3.fit_predict(X3_s)

# DBSCAN
db3 = DBSCAN(eps=0.3, min_samples=5)
labels_db3 = db3.fit_predict(X3_s)
n_clusters_db = len(set(labels_db3)) - (1 if -1 in labels_db3 else 0)
n_noise_db    = (labels_db3 == -1).sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# K-Means
for cl, cor in zip([0, 1], ['#2563eb', '#e11d48']):
    mask = labels_km3 == cl
    axes[0].scatter(X_with_out[mask, 0], X_with_out[mask, 1],
                    color=cor, alpha=.6, s=35)
axes[0].set_title('K-Means (K=2) — falha em clusters curvos\n'
                  '(mistura os dois grupos)')

# DBSCAN
cores_db = {-1: '#94a3b8', 0: '#2563eb', 1: '#e11d48', 2: '#15803d', 3: '#d97706'}
rotulos_db = {-1: f'Ruído/Outlier ({n_noise_db} pts)', 0: 'Cluster 0', 1: 'Cluster 1'}
for cl in sorted(set(labels_db3)):
    mask = labels_db3 == cl
    sz   = 80 if cl == -1 else 35
    mk   = 'x' if cl == -1 else 'o'
    axes[1].scatter(X_with_out[mask, 0], X_with_out[mask, 1],
                    color=cores_db.get(cl, '#000'), alpha=.7,
                    s=sz, marker=mk, label=rotulos_db.get(cl, f'Cluster {cl}'))
axes[1].set_title(f'DBSCAN (eps=0.3, min_samples=5)\n'
                  f'{n_clusters_db} clusters + {n_noise_db} outliers detectados')
axes[1].legend(fontsize=9)

for ax in axes:
    ax.set_xlabel('X1')
    ax.set_ylabel('X2')

plt.suptitle('Exemplo 3 — DBSCAN vs K-Means em dados curvos',
             fontweight='bold')
plt.tight_layout()
plt.show()

print(f'DBSCAN: {n_clusters_db} clusters encontrados, {n_noise_db} pontos de ruído')

In [ ]:
# ── Escolhendo eps via gráfico de k-vizinhos ──────────────────
nbrs = NearestNeighbors(n_neighbors=5).fit(X3_s)
dists3, _ = nbrs.kneighbors(X3_s)
dists_sorted = np.sort(dists3[:, -1])  # distância ao 5º vizinho

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(dists_sorted, color='#2563eb', lw=2)
ax.axhline(0.3, color='red', ls='--', lw=1.5, label='eps = 0,3 (cotovelo)')
ax.set_xlabel('Pontos ordenados')
ax.set_ylabel('Distância ao 5º vizinho mais próximo')
ax.set_title('Gráfico de k-vizinhos — escolha de eps\n'
             'Ponto de inflexão (cotovelo) = eps recomendado')
ax.legend()
ax.grid(alpha=.3)
plt.tight_layout()
plt.show()

### Como ler o gráfico de k-vizinhos para escolher eps

O gráfico acima mostra, para cada ponto dos dados, a **distância até seu 5º vizinho mais próximo**, ordenada da menor para a maior.

**Lógica:** se um ponto faz parte de uma região densa, seu 5º vizinho estará perto (distância baixa). Se o ponto está isolado (ruído), o 5º vizinho estará longe (distância alta).

No gráfico aparecem claramente **duas regiões**:
1. **Platô baixo à esquerda:** são os pontos dentro dos clusters — a distância ao vizinho é pequena e estável
2. **Salto brusco à direita:** são os pontos de ruído/outliers — a distância cresce rapidamente

O **ponto de inflexão** (cotovelo) entre o platô e o salto é o valor ideal de `eps`. Usar esse valor significa: "tudo que está dentro do padrão de densidade normal faz parte de um cluster; o que está além é ruído."

> Se `eps` for menor que o cotovelo — muitos pontos viram ruído (clusters se fragmentam)  
> Se `eps` for maior que o cotovelo — ruído entra nos clusters (perde sensibilidade a outliers)

---
## 4 · Exemplo 4 — Pipeline ML Completo com GridSearchCV
### Problema: diagnóstico de qualidade de vinho

**Variável alvo (Y):** vinho de alta qualidade (1) ou qualidade padrão (0)  
**Preditores (X):** acidez, álcool, açúcar residual, pH, teor de sulfatos

---

### O que são hiperparâmetros?

Todo algoritmo de ML tem dois tipos de parâmetros:

**Parâmetros** (aprendidos dos dados durante o treino):
- Coeficientes da regressão, pesos das conexões, thresholds das divisões
- O modelo os descobre sozinho — você nao define

**Hiperparâmetros** (definidos por você, antes do treino):
- `n_estimators` do Random Forest, `max_depth` da Arvore, `K` do KNN
- Controlam **como** o modelo aprende — são "configurações" do algoritmo

> **Analogia:** hiperparâmetros são como os ajustes de temperatura e tempo de uma receita. A receita em si (o algoritmo) é a mesma, mas pequenos ajustes fazem uma grande diferença no resultado.

---

### Como o GridSearchCV funciona?

O GridSearchCV testa **todas as combinações possíveis** de uma grade de hiperparâmetros, usando validação cruzada para avaliar cada uma.

```
Grade definida:
  n_estimators: [100, 200, 300]   --> 3 valores
  max_depth:    [None, 10, 20]    --> 3 valores
  min_samples_leaf: [1, 3, 5]    --> 3 valores

Total de combinações: 3 x 3 x 3 = 27

Para cada combinação, executa CV 5-fold:
  27 combinações x 5 folds = 135 modelos treinados no total

Retorna: a combinação com melhor métrica média nos folds de validação
```

> **Por isso usa validação cruzada interna:** se o GridSearch usasse o conjunto de teste para escolher hiperparâmetros, estaria "vendo o futuro" e a avaliação final seria otimista demais. O teste só é tocado UMA vez, no final, para medir o desempenho real.

---

### O que é Pipeline e por que é fundamental?

Um `Pipeline` encadeia etapas de processamento e o modelo em um único objeto:

```python
Pipeline([
    ('scaler', StandardScaler()),        # etapa 1: padronização
    ('clf',    RandomForestClassifier()) # etapa 2: modelo
])
```

**Por que nao simplesmente padronizar os dados antes?**

Imagine que você padroniza X inteiro (treino + teste) antes de dividir:

```
Errado:
   StandardScaler.fit(X_inteiro)    <- a média e o DP do TESTE vazam para o treino
   --> split em treino/teste
   --> modelo aprende com informação "do futuro"
   --> métricas otimistas (data leakage)

Correto com Pipeline:
   split em treino/teste
   --> dentro de cada fold do CV:
       StandardScaler.fit(X_treino_do_fold)     <- só vê o treino
       StandardScaler.transform(X_teste_do_fold) <- aplica sem vazar
```

Isso é chamado de **data leakage** (vazamento de dados) — um dos erros mais comuns em projetos de ML, que faz o modelo parecer melhor do que é na realidade.

---

### Sobre o dataset de vinho deste exemplo

Usamos variáveis físico-químicas do vinho para prever se ele é de alta qualidade. Faz sentido intuitivo:
- **Álcool alto** — geralmente associado a vinhos mais encorpados e complexos
- **Sulfatos moderados** — preservam o vinho e influenciam o sabor
- **Acidez baixa** — vinhos mais suaves ao paladar
- **Açúcar residual baixo** — vinhos secos tendem a ser mais valorizados por sommeliers

In [ ]:
# ── Dados de vinho ────────────────────────────────────────────
np.random.seed(99)
n4 = 600

acidez        = np.random.uniform(5.0, 9.5, n4)
alcool        = np.random.uniform(8.0, 14.5, n4)
acucar_res    = np.random.uniform(1.0, 15.0, n4)
ph            = np.random.uniform(2.9, 3.8, n4)
sulfatos      = np.random.uniform(0.3, 1.5, n4)

logit4 = (-6
          + 0.5  * alcool
          + 1.5  * sulfatos
          - 0.3  * acucar_res
          - 0.8  * acidez
          + 1.2  * ph)
prob4   = 1 / (1 + np.exp(-logit4))
qualidade_alta = (np.random.uniform(0, 1, n4) < prob4).astype(int)

df4 = pd.DataFrame({
    'acidez':     acidez.round(2),
    'alcool':     alcool.round(2),
    'acucar_res': acucar_res.round(2),
    'ph':         ph.round(2),
    'sulfatos':   sulfatos.round(3),
    'alta_qualidade': qualidade_alta
})

print('Distribuição das classes:')
print(df4['alta_qualidade'].value_counts().rename({0: 'Qualidade Padrão (0)', 1: 'Alta Qualidade (1)'}))
print(f'\nTaxa de alta qualidade: {df4["alta_qualidade"].mean():.1%}')
df4.head()

In [ ]:
# ── GridSearchCV — busca do melhor modelo ─────────────────────
X4 = df4.drop(columns='alta_qualidade')
y4 = df4['alta_qualidade']

X4_train, X4_test, y4_train, y4_test = train_test_split(
    X4, y4, test_size=0.2, random_state=42, stratify=y4
)

# Pipeline com Random Forest
pipe_rf4 = Pipeline([
    ('clf', RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    'clf__n_estimators': [100, 200, 300],
    'clf__max_depth':    [None, 10, 20],
    'clf__min_samples_leaf': [1, 3, 5],
}

cv4 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gs4 = GridSearchCV(
    pipe_rf4, param_grid_rf,
    cv=cv4,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)
gs4.fit(X4_train, y4_train)

print('Melhores hiperparâmetros encontrados:')
for param, val in gs4.best_params_.items():
    print(f'  {param}: {val}')
print(f'\nMelhor F1 (CV 5-fold): {gs4.best_score_:.4f}')

In [ ]:
# ── Avaliação FINAL no conjunto de teste ───────────────────────
y4_pred  = gs4.best_estimator_.predict(X4_test)
y4_proba = gs4.best_estimator_.predict_proba(X4_test)[:, 1]

print('Avaliação final no conjunto de teste:')
print(classification_report(y4_test, y4_pred,
                             target_names=['Padrão', 'Alta Qualidade']))
print(f'AUC = {roc_auc_score(y4_test, y4_proba):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Matriz de confusão
cm4 = confusion_matrix(y4_test, y4_pred)
disp4 = ConfusionMatrixDisplay(cm4, display_labels=['Padrão', 'Alta Qualidade'])
disp4.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matriz de Confusão — Conjunto de Teste')

# Importância de variáveis
rf4 = gs4.best_estimator_.named_steps['clf']
imp4 = pd.Series(rf4.feature_importances_, index=X4.columns).sort_values()
imp4.plot(kind='barh', ax=axes[1], color='#15803d', alpha=.85)
axes[1].set_title('Importância de Variáveis\n(Random Forest otimizado)')
axes[1].set_xlabel('Feature Importance')
axes[1].grid(axis='x', alpha=.3)

plt.suptitle('Exemplo 4 — Avaliação do modelo final (GridSearchCV)', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nInsights:')
print('  • GridSearchCV testou', len(gs4.cv_results_['mean_test_score']), 'combinações')
print(f'  • Melhores parâmetros: {gs4.best_params_}')
print(f'  • Variável mais importante: {imp4.idxmax()}')

---
## 5 · EXERCÍCIO — Sua Vez!

### Contexto
Uma operadora de plano de saúde quer entender melhor sua base de beneficiários.
Para isso, precisa de **duas análises complementares**:

1. **Classificação supervisionada:** prever quais beneficiários usarão serviços de alto custo no próximo mês
2. **Clusterização não supervisionada:** descobrir perfis de beneficiários para personalizar planos e campanhas

| Variável | Descrição |
|----------|-----------|
| `idade` | Idade do beneficiário |
| `consultas_mes` | Consultas médicas por mês |
| `exames_ano` | Exames realizados no último ano |
| `doencas_cronicas` | Número de doenças crônicas |
| `imc` | Índice de Massa Corporal |
| `alto_custo` | **Alvo para classificação** — 1 = usou serviços de alto custo |

### Tarefas de Classificação

1. **Explore os dados** — distribuições e correlações com `alto_custo`
2. **Treine 3 classificadores** — Árvore de Decisão, Random Forest e KNN — e compare via CV 5-fold
3. **Identifique o melhor modelo** e avalie no conjunto de teste (F1 + AUC)
4. **Interprete a importância de variáveis** do melhor modelo

### Tarefas de Clusterização

5. **Aplique K-Means** — use cotovelo + silhouette para escolher K
6. **Descreva cada cluster** — qual é o perfil de cada grupo de beneficiários?
7. **Aplique DBSCAN** — quantos outliers foram detectados? Quem são esses beneficiários?

> **Dica:** para a clusterização, use apenas as colunas de features (sem `alto_custo`).

In [ ]:
# ── Dados do exercício (não altere esta célula) ────────────────
np.random.seed(77)
n_ex = 500

idade_ex         = np.random.randint(18, 76, n_ex)
consultas_mes_ex = np.random.poisson(2, n_ex).clip(0, 10)
exames_ano_ex    = np.random.poisson(4, n_ex).clip(0, 20)
doencas_cr_ex    = np.random.randint(0, 5, n_ex)
imc_ex           = np.random.normal(27, 5, n_ex).clip(16, 45)

logit_ex2 = (-4
             + 0.03  * idade_ex
             + 0.25  * consultas_mes_ex
             + 0.10  * exames_ano_ex
             + 0.70  * doencas_cr_ex
             + 0.05  * imc_ex)
prob_ex2   = 1 / (1 + np.exp(-logit_ex2))
alto_custo = (np.random.uniform(0, 1, n_ex) < prob_ex2).astype(int)

df_ex2 = pd.DataFrame({
    'idade':          idade_ex,
    'consultas_mes':  consultas_mes_ex,
    'exames_ano':     exames_ano_ex,
    'doencas_cronicas': doencas_cr_ex,
    'imc':            imc_ex.round(1),
    'alto_custo':     alto_custo
})

print('Dados do exercício — primeiras linhas:')
print(df_ex2.head(10))
print(f'\nTotal de beneficiários: {len(df_ex2)}')
print(f'Taxa de alto custo: {df_ex2["alto_custo"].mean():.1%}')

In [ ]:
# ── TAREFA 1: Exploração dos dados ────────────────────────────
# Use .describe(), value_counts() e histogramas

# Seu código aqui:


In [ ]:
# ── TAREFAS 2 e 3: Comparar classificadores via CV ─────────────
# Dica: use Pipeline para cada modelo
# Compare via cross_val_score com scoring='f1' e 'roc_auc'

X_ex2 = df_ex2.drop(columns='alto_custo')
y_ex2 = df_ex2['alto_custo']

X_ex2_train, X_ex2_test, y_ex2_train, y_ex2_test = train_test_split(
    X_ex2, y_ex2, test_size=_____,  # <-- complete: 0.2
    random_state=42, stratify=y_ex2
)

# Defina os modelos aqui:
modelos_ex2 = {
    'Árvore':         Pipeline([('clf', DecisionTreeClassifier(max_depth=_____, random_state=42))]),
    'Random Forest':  Pipeline([('clf', RandomForestClassifier(n_estimators=200, random_state=42))]),
    'KNN (k=7)':      Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=_____))]),
}

# Seu código de comparação aqui:


In [ ]:
# ── TAREFA 4: Importância de variáveis do melhor modelo ────────
# Treine o melhor modelo (Random Forest) no treino e avalie no teste
# Plote as importâncias de features

# Seu código aqui:


In [ ]:
# ── TAREFA 5: K-Means — escolha do K ──────────────────────────
# Use apenas as features (sem alto_custo)
# Padronize com StandardScaler
# Plote cotovelo e silhouette para K de 2 a 8

X_cluster = df_ex2.drop(columns='alto_custo')

# Padronização
scaler_ex2 = StandardScaler()
X_cluster_s = scaler_ex2.fit_transform(X_cluster)

# Seu código de cotovelo e silhouette aqui:


In [ ]:
# ── TAREFA 6: K-Means — perfil dos clusters ────────────────────
# Treine com o K escolhido na tarefa anterior
# Calcule o perfil médio de cada cluster
# Visualize com PCA 2D

# Seu código aqui:


In [ ]:
# ── TAREFA 7: DBSCAN — outliers ───────────────────────────────
# Aplique DBSCAN nos dados padronizados X_cluster_s
# Dica: use eps entre 0.5 e 1.5, min_samples=5
# Quantos clusters e outliers foram encontrados?
# Quem são esses beneficiários atípicos?

# Seu código aqui:


---
### Perguntas para reflexão

Responda nas células abaixo (texto livre — clique duas vezes para editar):

**1.** Qual dos três classificadores teve melhor desempenho? Faz sentido dado o que vimos sobre cada algoritmo?

> *Sua resposta aqui...*

**2.** A variável mais importante para prever alto custo é a esperada clinicamente? Explique.

> *Sua resposta aqui...*

**3.** Os clusters do K-Means revelaram grupos interpretáveis de beneficiários? Como você nomearia cada perfil?

> *Sua resposta aqui...*

**4.** Os outliers detectados pelo DBSCAN têm características em comum? O que isso pode significar para a operadora de saúde?

> *Sua resposta aqui...*

**5.** Em que situação faz mais sentido usar **classificação** e em que situação faz mais sentido usar **clusterização** neste contexto de plano de saúde?

> *Sua resposta aqui...*

---
**Ciência de Dados · Aula 11 · Prof. Mesc. Diego Ramos Inácio · Univassouras · 2026**